In [2]:
#python

# inlegalbert_bilstm_mha_crf_twostage_v5.py  (TWO-STAGE LEARNING REVISION)
#
# Architecture:
#   InLegalBERT  →  BiLSTM  →  Multi-Head Attention Pooling  →  Linear  →  CRF
#
# TWO-STAGE TRAINING STRATEGY (new in v5):
# ─────────────────────────────────────────────────────────────────────────────
#
#   ┌─────────────────────────────────────────────────────────────────────┐
#   │  STAGE 1 — "Learn WHAT to distinguish"  (Representation Learning)  │
#   │                                                                     │
#   │  Goal  : Build class-discriminative representations for all 13      │
#   │          rhetorical roles, especially rare ones, BEFORE CRF learns  │
#   │          transitions.  If CRF trains on random embeddings it learns │
#   │          trivial majority-class transitions that are hard to unlearn.│
#   │                                                                     │
#   │  What trains : BERT (unfrozen layers) + BiLSTM + MHA + Classifier  │
#   │  What frozen : CRF layer  (CRF is NOT used in loss or decode)       │
#   │  Loss        : Strong class-weighted CE only (no CRF overhead)      │
#   │  Sampler     : WeightedRandomSampler — docs with rare classes are   │
#   │                oversampled proportional to their rarest label's     │
#   │                inverse frequency.  Rare classes seen much more.     │
#   │  LR          : Higher HEAD_LR (representations need fast learning)  │
#   │  Scheduler   : Cosine with warm-up                                  │
#   │  Early stop  : On weighted-F1 (good proxy for representation quality)│
#   └─────────────────────────────────────────────────────────────────────┘
#
#   ┌─────────────────────────────────────────────────────────────────────┐
#   │  STAGE 2 — "Learn HOW to sequence them"  (Sequence Refinement)     │
#   │                                                                     │
#   │  Goal  : Fine-tune the CRF to learn valid rhetorical transitions    │
#   │          given the now-discriminative stage-1 representations.      │
#   │          Fine-tune the full model end-to-end at a lower LR.        │
#   │                                                                     │
#   │  What trains : Full model — all unfrozen BERT layers + head + CRF  │
#   │  What frozen : Nothing extra (CRF now active)                       │
#   │  Loss        : CRF loss + mild class-weighted CE (as in v3)         │
#   │  Sampler     : Mild balanced sampling (less aggressive than S1)     │
#   │  LR          : Lower HEAD_LR and BERT_LR (avoid destroying S1 reps)│
#   │  Scheduler   : Cosine with warm-up (fresh schedule from epoch 0)    │
#   │  SWA         : Applied in final 20% of stage 2                      │
#   │  Early stop  : On macro-F1 (sequence-level quality)                 │
#   └─────────────────────────────────────────────────────────────────────┘
#
# All v3 features preserved: layer-wise LR decay, SWA, cosine scheduler,
# gradient accumulation, gradient clipping, CRF shape-trimming fix.
#

import os, json, random, time
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from transformers import (
    AutoTokenizer, AutoModel,
    get_cosine_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)

# ═══════════════════════════════════════════════════════════
# CONFIG  (v3 base — shared across both stages)
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH     = "dataset/build_train.jsonl"
DEV_PATH       = "dataset/build_dev.jsonl"
TEST_PATH      = "dataset/build_test.jsonl"
OUT_DIR        = "rrc_bilstm_mha_crf_twostage_v5_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

SEED           = 42
MAX_SEQ_LENGTH = 32
RARE_THRESHOLD = 0.05

BERT_FREEZE_LAYERS = 10
BERT_LR_DECAY      = 0.9
GRAD_CLIP          = 1.0
LABEL_SMOOTHING    = 0.1
WARMUP_RATIO       = 0.05

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"

# ── BiLSTM / MHA dims (shared) ─────────────────────────────
SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 1
MHA_HEADS        = 4
MHA_DROPOUT      = 0.1
CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 1

# ═══════════════════════════════════════════════════════════
# STAGE-1 CONFIG  —  Representation Learning
# ═══════════════════════════════════════════════════════════
S1_EPOCHS          = 30         # Max stage-1 epochs (ES may stop earlier)
S1_BATCH_DOCS      = 2
S1_BERT_LR         = 1e-5
S1_HEAD_LR         = 5e-4       # Aggressive — representations need to move fast
S1_WEIGHT_DECAY    = 0.05
S1_DROPOUT         = 0.3        # Lower dropout so minority classes still activate
S1_GRAD_ACC        = 2
S1_AUX_CE_WEIGHT   = 1.0        # Stage 1 IS the CE loss (no CRF)
S1_ES_PATIENCE     = 6          # Stop S1 early → move to S2
S1_ES_MIN_DELTA    = 1e-4
S1_ES_METRIC       = "weighted_f1"   # Weighted F1: good proxy for representation quality
# Oversampling strength: doc weight = freq^(-S1_OVERSAMPLE_POWER)
# 1.0 = inverse freq, 0.5 = sqrt inverse, 0.0 = uniform
S1_OVERSAMPLE_POWER = 0.75

# ═══════════════════════════════════════════════════════════
# STAGE-2 CONFIG  —  Sequence Refinement
# ═══════════════════════════════════════════════════════════
S2_EPOCHS          = 50         # Max stage-2 epochs (ES may stop earlier)
S2_BATCH_DOCS      = 2
S2_BERT_LR         = 5e-6       # Much lower — don't destroy stage-1 representations
S2_HEAD_LR         = 1e-4       # Also lower
S2_WEIGHT_DECAY    = 0.1
S2_DROPOUT         = 0.5        # Higher dropout for regularisation
S2_GRAD_ACC        = 4
S2_AUX_CE_WEIGHT   = 0.3        # CRF is primary; CE is auxiliary
S2_ES_PATIENCE     = 8
S2_ES_MIN_DELTA    = 1e-4
S2_ES_METRIC       = "macro_f1"      # Macro F1: sequence-level quality incl. rare classes
S2_SWA_START_FRAC  = 0.80
S2_SWA_LR          = 5e-5
# Mild oversampling in S2 (less aggressive than S1)
S2_OVERSAMPLE_POWER = 0.40


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING  (parameterised)
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience: int, min_delta: float = 1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# PARAMETER COUNTER
# ═══════════════════════════════════════════════════════════
def count_parameters(model):
    component_map = {
        "bert":           "InLegalBERT Encoder",
        "sent_bilstm":    "Sentence BiLSTM",
        "mha_pooling":    "Multi-Head Attn Pooling",
        "ctx_bilstm":     "Context BiLSTM",
        "classifier":     "Classifier Head",
        "crf":            "CRF",
    }
    rows = []
    for attr, name in component_map.items():
        module = getattr(model, attr, None)
        if module is None:
            continue
        trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
        frozen    = sum(p.numel() for p in module.parameters() if not p.requires_grad)
        rows.append({"Component": name, "Trainable Params": trainable,
                     "Frozen Params": frozen, "Total Params": trainable + frozen})
    total_t = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_f = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    rows.append({"Component": "── TOTAL ──", "Trainable Params": total_t,
                 "Frozen Params": total_f, "Total Params": total_t + total_f})
    print("\n" + "=" * 70)
    print("MODEL PARAMETER SUMMARY  (v5 Two-Stage)")
    print("=" * 70)
    print(f"  {'Component':<30} {'Trainable':>14} {'Frozen':>10} {'Total':>12}")
    print("-" * 70)
    for r in rows:
        if r["Component"] == "── TOTAL ──":
            print("─" * 70)
        print(f"  {r['Component']:<30} {r['Trainable Params']:>14,} "
              f"{r['Frozen Params']:>10,} {r['Total Params']:>12,}")
    print("=" * 70)
    return total_t, total_f, rows


# ═══════════════════════════════════════════════════════════
# RARE-CLASS DETECTION
# ═══════════════════════════════════════════════════════════
def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]
    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({counts.get(label2id[lbl], 0):5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# CLASS WEIGHTS  (for CE loss)
# ═══════════════════════════════════════════════════════════
def compute_class_weights(docs: list) -> torch.Tensor:
    """Smooth inverse-frequency weights; rare classes → higher weight."""
    all_labels = [lab for _, labs in docs for lab in labs]
    counts = Counter(all_labels)
    total  = sum(counts.values())
    weights = torch.tensor(
        [1.0 / (counts.get(i, 0) / max(total, 1) + 1e-5) for i in range(NUM_LABELS)],
        dtype=torch.float,
    )
    weights = weights / weights.mean()   # normalise → mean weight = 1
    return weights


# ═══════════════════════════════════════════════════════════
# DOCUMENT SAMPLER  (for balanced DataLoader)
# ═══════════════════════════════════════════════════════════
def build_doc_sampler(
    docs: list,
    label_freqs: dict,
    power: float,
) -> WeightedRandomSampler:
    """
    Weight each document by the rarest label it contains, raised to `power`.
    power=1.0  → strict inverse freq (rare docs up-weighted a lot)
    power=0.5  → square-root (gentler)
    power=0.0  → uniform (no oversampling)

    Intuition: documents that contain at least one ISSUE / PRE_NOT_RELIED
    sentence are sampled much more often so the model sees those classes
    repeatedly during representation learning.
    """
    doc_weights = []
    for _, labs in docs:
        # Weight = min label frequency in this doc (rarest class drives it)
        min_freq = min(label_freqs.get(id2label[l], 1.0) for l in labs)
        # Inverse frequency, raised to power
        w = (1.0 / (min_freq + 1e-8)) ** power
        doc_weights.append(w)
    doc_weights = torch.tensor(doc_weights, dtype=torch.double)
    sampler = WeightedRandomSampler(
        weights     = doc_weights,
        num_samples = len(doc_weights),
        replacement = True,
    )
    rare_docs = sum(1 for w in doc_weights if w > doc_weights.mean())
    print(f"   📐 Doc sampler (power={power:.2f}): "
          f"{rare_docs}/{len(docs)} docs above mean weight | "
          f"max/min ratio={doc_weights.max()/doc_weights.min():.1f}x")
    return sampler


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        all_docs.append((sents[:max_sents], labs[:max_sents]))
    return all_docs


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding    = "max_length",
            truncation = True,
            max_length = self.max_length,
            return_tensors = "pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get("token_type_ids",
                                      torch.zeros_like(enc["input_ids"])),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)
    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)
    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t
    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.query     = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)
        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x, key_padding_mask=None):
        N, L, H = x.shape
        K = self.key_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.val_proj(x).view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            attn = attn.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2), -1e9)
        attn    = self.attn_drop(F.softmax(attn, dim=-1))
        context = torch.matmul(attn, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


# ═══════════════════════════════════════════════════════════
# MODEL
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):

    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = S2_DROPOUT,     # Set per-stage via set_dropout()
        freeze_layers    = BERT_FREEZE_LAYERS,
        class_weights_s1 = None,           # Strong weights for stage-1 CE
        class_weights_s2 = None,           # Mild weights for stage-2 CE
    ):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size
        self.dropout  = nn.Dropout(dropout)
        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size    = self.bert_dim,
            hidden_size   = sent_lstm_hidden,
            num_layers    = sent_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if sent_lstm_layers > 1 else 0.0,
        )
        sent_out_dim = sent_lstm_hidden * 2      # 256

        self.mha_pooling     = MultiHeadAttentionPooling(sent_out_dim, mha_heads, mha_dropout)
        self.sent_layer_norm = nn.LayerNorm(sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size    = sent_out_dim,
            hidden_size   = ctx_lstm_hidden,
            num_layers    = ctx_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if ctx_lstm_layers > 1 else 0.0,
        )
        ctx_out_dim = ctx_lstm_hidden * 2        # 128

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(ctx_out_dim, ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ctx_out_dim // 2, num_labels),
        )
        self.crf = CRF(num_tags=num_labels, batch_first=True)

        # Separate CE losses for each stage (different weights)
        self.ce_s1 = nn.CrossEntropyLoss(
            weight          = class_weights_s1,
            label_smoothing = LABEL_SMOOTHING,
            ignore_index    = -100,
        )
        self.ce_s2 = nn.CrossEntropyLoss(
            weight          = class_weights_s2,
            label_smoothing = LABEL_SMOOTHING,
            ignore_index    = -100,
        )
        # Active training stage: 1 or 2
        self._stage = 1

    # ----------------------------------------------------------
    def set_stage(self, stage: int):
        """Switch between stage 1 (CE-only) and stage 2 (CRF+CE)."""
        assert stage in (1, 2)
        self._stage = stage
        # Freeze / unfreeze CRF
        for p in self.crf.parameters():
            p.requires_grad = (stage == 2)
        print(f"\n🔀 Switched to Stage {stage} | "
              f"CRF {'trainable' if stage == 2 else 'frozen (CE-only loss)'}")

    def set_dropout(self, dropout: float):
        """Hot-swap dropout rates between stages without re-instantiating."""
        self.dropout.p = dropout
        for m in self.modules():
            if isinstance(m, nn.Dropout) and m is not self.dropout:
                m.p = dropout
        print(f"   Dropout set to {dropout}")

    # ----------------------------------------------------------
    def _freeze_bert_layers(self, n_freeze: int):
        for p in self.bert.embeddings.parameters():
            p.requires_grad = False
        enc = self.bert.encoder.layer
        for i in range(min(n_freeze, len(enc))):
            for p in enc[i].parameters():
                p.requires_grad = False
        print(f"\n❄️  BERT frozen: embeddings + layers 0-{n_freeze-1}.")
        print(f"🔥 BERT trainable: layers {n_freeze}-{len(enc)-1} "
              f"({len(enc)-n_freeze} layers) + pooler.\n")

    # ----------------------------------------------------------
    def encode_sentences(self, input_ids, attention_mask, token_type_ids, lengths=None):
        B, T, L = input_ids.shape
        N = B * T
        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)
        valid = flat_mask.sum(dim=-1) > 0
        embs  = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)
        if valid.any():
            out = self.bert(
                input_ids      = flat_ids[valid],
                attention_mask = flat_mask[valid],
                token_type_ids = flat_types[valid],
            )
            embs[valid] = out.last_hidden_state.to(embs.dtype)
        embs      = self.dropout(embs)
        lstm_out, _ = self.sent_bilstm(embs)
        lstm_out    = self.dropout(lstm_out)
        pad_mask    = (flat_mask == 0).clone()
        pad_mask[~valid] = False
        sent_vecs   = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs   = self.sent_layer_norm(sent_vecs)
        sent_vecs   = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)
        return sent_vecs.view(B, T, -1)

    # ----------------------------------------------------------
    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        sent_vecs = self.encode_sentences(input_ids, attention_mask,
                                          token_type_ids, lengths=lengths)
        sent_vecs = self.dropout(sent_vecs)

        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _    = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True)
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs)

        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)

        # ── Shape-safety: trim labels to emissions' temporal dim ──
        # After pack→unpack, emissions.shape[1] == actual doc length,
        # but labels.shape[1] == T_max_in_batch.  Trim to avoid CRF crash.
        T_emit = emissions.shape[1]

        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :min(int(l.item()), T)] = True
        elif labels is not None:
            mask = (labels[:, :T_emit] != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)

        if labels is not None:
            labels_t = labels[:, :T_emit]         # trimmed labels
            B2, T2, C = emissions.shape
            flat_em  = emissions.reshape(B2 * T2, C)
            flat_lab = labels_t.reshape(B2 * T2)

            if self._stage == 1:
                # ── Stage 1: CE-only (no CRF) ──────────────────
                loss = self.ce_s1(flat_em, flat_lab)

            else:
                # ── Stage 2: CRF + weighted CE ─────────────────
                safe_lab = labels_t.clone()
                safe_lab[safe_lab == -100] = 0
                crf_loss = -self.crf(emissions, safe_lab,
                                     mask=mask, reduction="mean")
                ce_loss  = self.ce_s2(flat_em, flat_lab)
                loss     = crf_loss + S2_AUX_CE_WEIGHT * ce_loss

            return loss, emissions

        else:
            # ── Inference ─────────────────────────────────────
            if self._stage == 1:
                # Argmax decode (no CRF in stage 1)
                preds = emissions.argmax(dim=-1)
                result = []
                if lengths is not None:
                    for i, l in enumerate(lengths):
                        result.append(preds[i, :int(l.item())].cpu().tolist())
                else:
                    result = preds.cpu().tolist()
                return result, emissions
            else:
                return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# METRICS
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_prec  = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec  = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    wprecision  = precision_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_rec   = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec   = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    wrec        = recall_score(all_trues, all_preds, average="weighted", zero_division=0)
    acc         = accuracy_score(all_trues, all_preds)

    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)), average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)), average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)), average=None, zero_division=0)
    per_class_metrics = {
        id2label[i]: {"f1": float(per_class_f1[i]),
                      "precision": float(per_class_prec[i]),
                      "recall":    float(per_class_rec[i])}
        for i in range(NUM_LABELS)
    }

    present_rare = [r for r in rare_ids if r in set(all_trues)]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare, average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare, average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare, average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0

    cls_report = classification_report(
        [id2label[x] for x in all_trues],
        [id2label[x] for x in all_preds],
        labels=LABELS, digits=4, zero_division=0,
    )
    cm = confusion_matrix(
        [id2label[x] for x in all_trues],
        [id2label[x] for x in all_preds],
        labels=LABELS,
    )
    return {
        "macro_f1": macro_f1, "micro_f1": micro_f1, "weighted_f1": weighted_f1,
        "macro_precision": macro_prec, "micro_precision": micro_prec, "weighted_precision": wprecision,
        "macro_recall": macro_rec, "micro_recall": micro_rec, "weighted_recall": wrec,
        "rare_f1": rare_f1, "rare_precision": rare_prec, "rare_recall": rare_rec,
        "per_class_metrics": per_class_metrics,
        "accuracy": acc, "cls_report": cls_report, "cm": cm,
        "all_preds": all_preds, "all_trues": all_trues,
    }


# ═══════════════════════════════════════════════════════════
# TRAINER
# ═══════════════════════════════════════════════════════════
class TwoStageTrainer:

    def __init__(self, model: InLegalBERT_BiLSTM_MHA_CRF, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    # ----------------------------------------------------------
    # Optimizer builders  (separate per stage)
    # ----------------------------------------------------------
    def _build_optimizer(self, bert_lr: float, head_lr: float,
                         weight_decay: float, stage: int) -> torch.optim.AdamW:
        param_groups = []

        # BERT pooler
        param_groups.append({"params": list(self.model.bert.pooler.parameters()),
                              "lr": bert_lr, "weight_decay": weight_decay})

        # BERT encoder layers (top-down decay)
        enc = self.model.bert.encoder.layer
        n   = len(enc)
        for i in range(n - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth = (n - 1) - i
            params = [p for p in enc[i].parameters() if p.requires_grad]
            if params:
                param_groups.append({
                    "params":       params,
                    "lr":           bert_lr * (BERT_LR_DECAY ** depth),
                    "weight_decay": weight_decay,
                })

        # Head modules
        head_mods = [self.model.sent_bilstm, self.model.mha_pooling,
                     self.model.sent_layer_norm, self.model.ctx_bilstm,
                     self.model.classifier]
        if stage == 2:
            head_mods.append(self.model.crf)    # CRF only in stage 2

        head_params = []
        for m in head_mods:
            head_params.extend(list(m.parameters()))
        param_groups.append({"params": head_params,
                             "lr": head_lr, "weight_decay": weight_decay})
        return torch.optim.AdamW(param_groups)

    # ----------------------------------------------------------
    # Shared evaluation loop
    # ----------------------------------------------------------
    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False, model_override=None):
        m = model_override if model_override is not None else self.model
        m.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        t0 = time.time() if measure_inference_time else None

        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                lengths        = lengths.to(self.device)
                decoded, _ = m(input_ids, attention_mask, token_type_ids,
                               labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    all_preds.extend(seq_preds)
                    all_trues.extend(labels[i, :true_len].cpu().numpy().tolist())

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)

        if measure_inference_time:
            elapsed = time.time() - t0
            n_sents = len(all_trues)
            n_docs  = len(dataset)
            infer   = {
                "split": split_name, "n_documents": n_docs, "n_sentences": n_sents,
                "total_inference_time_s":     elapsed,
                "latency_per_document_ms":    elapsed / max(1, n_docs) * 1000,
                "latency_per_sentence_ms":    elapsed / max(1, n_sents) * 1000,
                "throughput_sentences_per_s": n_sents / max(1e-9, elapsed),
            }
            with open(os.path.join(OUT_DIR, f"inference_time_{split_name}.json"), "w") as f:
                json.dump(infer, f, indent=2)
            print(f"⏱  Inference ({split_name}): {elapsed:.2f}s | "
                  f"{infer['throughput_sentences_per_s']:.1f} sent/s")
            metrics["inference_time_info"] = infer

        return metrics

    def compute_val_loss(self, dataset, model_override=None):
        m = model_override if model_override is not None else self.model
        m.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_rrc)
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)
                loss, _ = m(input_ids, attention_mask, token_type_ids,
                            labels=labels, lengths=lengths)
                if not torch.isnan(loss):
                    total_loss += loss.item(); n += 1
        return total_loss / max(1, n)

    # ----------------------------------------------------------
    # STAGE 1  — Representation Learning
    # ----------------------------------------------------------
    def train_stage1(
        self,
        train_dataset: Dataset,
        dev_dataset:   Dataset,
        rare_ids:      list,
        docs:          list,
        label_freqs:   dict,
    ) -> list:
        """
        Returns history rows (list of dicts).
        CRF is FROZEN.  Loss = strong-weighted CE only.
        Balanced sampler gives rare-class documents more exposure.
        """
        print("\n" + "╔" + "═"*62 + "╗")
        print("║  STAGE 1 — Representation Learning (CE-only, Balanced)  ║")
        print("╚" + "═"*62 + "╝")
        print(f"  Epochs     : up to {S1_EPOCHS}  (ES patience={S1_ES_PATIENCE})")
        print(f"  BERT LR    : {S1_BERT_LR}    HEAD LR: {S1_HEAD_LR}")
        print(f"  Dropout    : {S1_DROPOUT}    Oversample power: {S1_OVERSAMPLE_POWER}")
        print(f"  ES metric  : {S1_ES_METRIC}")

        self.model.set_stage(1)
        self.model.set_dropout(S1_DROPOUT)

        sampler = build_doc_sampler(docs, label_freqs, S1_OVERSAMPLE_POWER)
        loader  = DataLoader(train_dataset, batch_size=S1_BATCH_DOCS,
                             sampler=sampler, collate_fn=collate_rrc)

        optimizer = self._build_optimizer(S1_BERT_LR, S1_HEAD_LR, S1_WEIGHT_DECAY, stage=1)
        total_steps  = len(loader) * S1_EPOCHS // S1_GRAD_ACC
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

        early_stopper = EarlyStopping(S1_ES_PATIENCE, S1_ES_MIN_DELTA)
        history       = []
        best_f1       = -1.0
        best_state    = None
        t_start       = time.time()

        for epoch in range(1, S1_EPOCHS + 1):
            self.model.train()
            running_loss, n_steps, nan_steps = 0.0, 0, 0
            t_ep = time.time()
            optimizer.zero_grad()

            for step, (input_ids, attention_mask,
                        token_type_ids, labels, lengths) in enumerate(loader):
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)

                loss, _ = self.model(input_ids, attention_mask, token_type_ids,
                                     labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss):
                    nan_steps += 1; optimizer.zero_grad(); continue

                (loss / S1_GRAD_ACC).backward()
                if (step + 1) % S1_GRAD_ACC == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                    optimizer.step(); scheduler.step(); optimizer.zero_grad()
                running_loss += loss.item(); n_steps += 1

            # Flush remaining
            if n_steps % S1_GRAD_ACC != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            avg_loss    = running_loss / max(1, n_steps)
            val_loss    = self.compute_val_loss(dev_dataset)
            val_metrics = self.evaluate(dev_dataset, rare_ids)
            monitor     = val_metrics[S1_ES_METRIC]

            nan_tag = f" [nan={nan_steps}]" if nan_steps else ""
            print(f"  S1 Ep {epoch:03d}/{S1_EPOCHS} | "
                  f"train: {avg_loss:.4f} | val: {val_loss:.4f} | "
                  f"macro_f1: {val_metrics['macro_f1']:.4f} | "
                  f"weighted_f1: {val_metrics['weighted_f1']:.4f} | "
                  f"rare_f1: {val_metrics['rare_f1']:.4f} | "
                  f"ES: {early_stopper.counter}/{early_stopper.patience}{nan_tag}")

            row = {
                "stage": 1, "epoch": epoch,
                "train_loss": avg_loss, "val_loss": val_loss,
                "val_accuracy": val_metrics["accuracy"],
                "val_macro_f1": val_metrics["macro_f1"],
                "val_micro_f1": val_metrics["micro_f1"],
                "val_weighted_f1": val_metrics["weighted_f1"],
                "val_rare_f1": val_metrics["rare_f1"],
                "val_macro_precision": val_metrics["macro_precision"],
                "val_micro_precision": val_metrics["micro_precision"],
                "val_weighted_precision": val_metrics["weighted_precision"],
                "val_rare_precision": val_metrics["rare_precision"],
                "val_macro_recall": val_metrics["macro_recall"],
                "val_micro_recall": val_metrics["micro_recall"],
                "val_weighted_recall": val_metrics["weighted_recall"],
                "val_rare_recall": val_metrics["rare_recall"],
                "epoch_train_time_s": time.time() - t_ep,
                "nan_steps": nan_steps,
                "swa_active": False,
                "timestamp": datetime.utcnow().isoformat(),
            }
            history.append(row)

            if monitor > best_f1 + S1_ES_MIN_DELTA:
                best_f1    = monitor
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"    ✔ New best S1 {S1_ES_METRIC}={best_f1:.4f}")

            if early_stopper.step(monitor):
                print(f"\n⏹  Stage 1 early stop at epoch {epoch}.\n")
                break

        s1_time = time.time() - t_start
        print(f"\n⏱  Stage 1 done: {s1_time/60:.2f} min | "
              f"best {S1_ES_METRIC}={best_f1:.4f}")

        # Restore best S1 weights before handing off to S2
        if best_state:
            self.model.load_state_dict(best_state)
            print("  Loaded best S1 checkpoint for Stage 2 initialisation.")

        return history, s1_time

    # ----------------------------------------------------------
    # STAGE 2  — Sequence Refinement
    # ----------------------------------------------------------
    def train_stage2(
        self,
        train_dataset: Dataset,
        dev_dataset:   Dataset,
        rare_ids:      list,
        docs:          list,
        label_freqs:   dict,
        tokenizer,
    ) -> list:
        """
        Returns history rows.
        CRF is ACTIVE.  Loss = CRF + mild class-weighted CE.
        Mild balanced sampler.  SWA in final 20% of stage.
        """
        print("\n" + "╔" + "═"*62 + "╗")
        print("║  STAGE 2 — Sequence Refinement  (CRF + CE, mild balance) ║")
        print("╚" + "═"*62 + "╝")
        print(f"  Epochs     : up to {S2_EPOCHS}  (ES patience={S2_ES_PATIENCE})")
        print(f"  BERT LR    : {S2_BERT_LR}    HEAD LR: {S2_HEAD_LR}")
        print(f"  Dropout    : {S2_DROPOUT}    Oversample power: {S2_OVERSAMPLE_POWER}")
        print(f"  ES metric  : {S2_ES_METRIC}  AUX_CE_WEIGHT: {S2_AUX_CE_WEIGHT}")

        self.model.set_stage(2)
        self.model.set_dropout(S2_DROPOUT)

        sampler = build_doc_sampler(docs, label_freqs, S2_OVERSAMPLE_POWER)
        loader  = DataLoader(train_dataset, batch_size=S2_BATCH_DOCS,
                             sampler=sampler, collate_fn=collate_rrc)

        optimizer = self._build_optimizer(S2_BERT_LR, S2_HEAD_LR, S2_WEIGHT_DECAY, stage=2)
        total_steps  = len(loader) * S2_EPOCHS // S2_GRAD_ACC
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

        swa_model    = AveragedModel(self.model)
        swa_start_ep = max(1, int(S2_EPOCHS * S2_SWA_START_FRAC))
        swa_sched    = SWALR(optimizer, swa_lr=S2_SWA_LR,
                             anneal_epochs=5, anneal_strategy="cos")
        swa_active   = False
        print(f"  SWA starts at S2 epoch {swa_start_ep}/{S2_EPOCHS}")

        early_stopper = EarlyStopping(S2_ES_PATIENCE, S2_ES_MIN_DELTA)
        history       = []
        best_f1       = -1.0
        best_state    = None
        t_start       = time.time()

        for epoch in range(1, S2_EPOCHS + 1):
            self.model.train()
            running_loss, n_steps, nan_steps = 0.0, 0, 0
            t_ep = time.time()
            optimizer.zero_grad()

            for step, (input_ids, attention_mask,
                        token_type_ids, labels, lengths) in enumerate(loader):
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)

                loss, _ = self.model(input_ids, attention_mask, token_type_ids,
                                     labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss):
                    nan_steps += 1; optimizer.zero_grad(); continue

                (loss / S2_GRAD_ACC).backward()
                if (step + 1) % S2_GRAD_ACC == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                    optimizer.step()
                    if not swa_active:
                        scheduler.step()
                    optimizer.zero_grad()
                running_loss += loss.item(); n_steps += 1

            if n_steps % S2_GRAD_ACC != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step()
                if not swa_active:
                    scheduler.step()
                optimizer.zero_grad()

            if epoch >= swa_start_ep:
                swa_active = True
                swa_model.update_parameters(self.model)
                swa_sched.step()

            avg_loss    = running_loss / max(1, n_steps)
            val_loss    = self.compute_val_loss(dev_dataset)
            val_metrics = self.evaluate(dev_dataset, rare_ids)
            monitor     = val_metrics[S2_ES_METRIC]

            nan_tag = f" [nan={nan_steps}]" if nan_steps else ""
            swa_tag = " [SWA]" if swa_active else ""
            print(f"  S2 Ep {epoch:03d}/{S2_EPOCHS} | "
                  f"train: {avg_loss:.4f} | val: {val_loss:.4f} | "
                  f"macro_f1: {val_metrics['macro_f1']:.4f} | "
                  f"rare_f1: {val_metrics['rare_f1']:.4f} | "
                  f"acc: {val_metrics['accuracy']:.4f} | "
                  f"ES: {early_stopper.counter}/{early_stopper.patience}"
                  f"{swa_tag}{nan_tag}")

            row = {
                "stage": 2, "epoch": epoch,
                "train_loss": avg_loss, "val_loss": val_loss,
                "val_accuracy": val_metrics["accuracy"],
                "val_macro_f1": val_metrics["macro_f1"],
                "val_micro_f1": val_metrics["micro_f1"],
                "val_weighted_f1": val_metrics["weighted_f1"],
                "val_rare_f1": val_metrics["rare_f1"],
                "val_macro_precision": val_metrics["macro_precision"],
                "val_micro_precision": val_metrics["micro_precision"],
                "val_weighted_precision": val_metrics["weighted_precision"],
                "val_rare_precision": val_metrics["rare_precision"],
                "val_macro_recall": val_metrics["macro_recall"],
                "val_micro_recall": val_metrics["micro_recall"],
                "val_weighted_recall": val_metrics["weighted_recall"],
                "val_rare_recall": val_metrics["rare_recall"],
                "epoch_train_time_s": time.time() - t_ep,
                "nan_steps": nan_steps,
                "swa_active": swa_active,
                "timestamp": datetime.utcnow().isoformat(),
            }
            history.append(row)

            if monitor > best_f1 + S2_ES_MIN_DELTA:
                best_f1    = monitor
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"    ✔ New best S2 {S2_ES_METRIC}={best_f1:.4f}")

            if early_stopper.step(monitor):
                print(f"\n⏹  Stage 2 early stop at epoch {epoch}.\n")
                break

        # ── Finalize SWA ───────────────────────────────────
        if swa_active:
            print("📐 Updating SWA BatchNorm stats...")
            update_bn(DataLoader(train_dataset, batch_size=S2_BATCH_DOCS,
                                 shuffle=False, collate_fn=collate_rrc),
                      swa_model, device=self.device)
            swa_metrics = self.evaluate(dev_dataset, rare_ids, model_override=swa_model)
            print(f"  SWA macro_f1: {swa_metrics['macro_f1']:.4f}")
            if swa_metrics["macro_f1"] > best_f1:
                best_f1    = swa_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in swa_model.module.state_dict().items()}
                print(f"  ✔ SWA beats individual checkpoint — using SWA model.")

        s2_time = time.time() - t_start
        print(f"\n⏱  Stage 2 done: {s2_time/60:.2f} min | "
              f"best {S2_ES_METRIC}={best_f1:.4f}")

        # Save best S2 checkpoint
        if best_state:
            self.model.load_state_dict(best_state)
            self._save_model_hf(best_state, tokenizer)

        return history, s2_time

    # ----------------------------------------------------------
    def _save_model_hf(self, state_dict, tokenizer):
        self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
        tokenizer.save_pretrained(BEST_MODEL_DIR)
        torch.save(state_dict, os.path.join(BEST_MODEL_DIR, "pytorch_model.bin"))
        with open(os.path.join(BEST_MODEL_DIR, "model_args.json"), "w") as f:
            json.dump({
                "bert_model_name": INLEGALBERT_MODEL_NAME,
                "sent_lstm_hidden": SENT_LSTM_HIDDEN,
                "sent_lstm_layers": SENT_LSTM_LAYERS,
                "ctx_lstm_hidden": CTX_LSTM_HIDDEN,
                "ctx_lstm_layers": CTX_LSTM_LAYERS,
                "mha_heads": MHA_HEADS,
                "mha_dropout": MHA_DROPOUT,
                "num_labels": NUM_LABELS,
                "labels": LABELS,
                "label2id": label2id,
                "id2label": id2label,
                "max_seq_length": MAX_SEQ_LENGTH,
                "freeze_layers": BERT_FREEZE_LAYERS,
                "two_stage": True,
            }, f, indent=2)
        print(f"\n💾 Best model saved → {BEST_MODEL_DIR}/")


# ═══════════════════════════════════════════════════════════
# PLOTTING
# ═══════════════════════════════════════════════════════════
def plot_history(hist_df: pd.DataFrame):
    # Separate by stage
    s1 = hist_df[hist_df["stage"] == 1].copy()
    s2 = hist_df[hist_df["stage"] == 2].copy()

    # Give S2 a continuous x-axis starting after S1
    s2_offset = len(s1)
    s1["global_epoch"] = s1["epoch"]
    s2["global_epoch"] = s2["epoch"] + s2_offset
    full = pd.concat([s1, s2], ignore_index=True)
    s1_end = s1["global_epoch"].max() if len(s1) else 0

    swa_rows = s2[s2["swa_active"] == True]
    swa_ep   = int(swa_rows["global_epoch"].iloc[0]) if len(swa_rows) else None

    def _add_stage_divider(ax):
        if s1_end > 0:
            ax.axvline(s1_end + 0.5, color="purple", linestyle=":",
                       alpha=0.7, linewidth=1.5, label="Stage 1→2")
        if swa_ep:
            ax.axvline(swa_ep, color="green", linestyle="--",
                       alpha=0.6, linewidth=1.5, label=f"SWA starts")

    # ── 1. Loss ────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(full["global_epoch"], full["train_loss"], label="Train Loss", marker="o", markersize=3)
    ax.plot(full["global_epoch"], full["val_loss"],   label="Val Loss",   marker="s", markersize=3)
    _add_stage_divider(ax)
    ax.set_title("Two-Stage Training: Loss Curves")
    ax.set_xlabel("Epoch (global)"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    p = os.path.join(OUT_DIR, "twostage_loss_curve.png")
    plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

    # ── 2. F1 curves ───────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for col, lbl, ls in [
        ("val_macro_f1",    "Macro-F1",    "-"),
        ("val_micro_f1",    "Micro-F1",    "--"),
        ("val_weighted_f1", "Weighted-F1", "-."),
        ("val_rare_f1",     "Rare-F1",     ":"),
    ]:
        axes[0].plot(full["global_epoch"], full[col], label=lbl, linestyle=ls)
    _add_stage_divider(axes[0])
    axes[0].set_title("Validation F1 — Two Stages")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(full["global_epoch"], full["train_loss"], label="Train", marker="o", markersize=3)
    axes[1].plot(full["global_epoch"], full["val_loss"],   label="Val",   marker="s", markersize=3)
    _add_stage_divider(axes[1])
    axes[1].set_title("Loss — Two Stages")
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    p = os.path.join(OUT_DIR, "twostage_f1_curve.png")
    plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

    # ── 3. Precision / Recall ──────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for col, lbl, ls in [
        ("val_macro_precision", "Macro-P", "-"), ("val_micro_precision", "Micro-P", "--"),
        ("val_weighted_precision", "W-P", "-."), ("val_rare_precision", "Rare-P", ":"),
    ]:
        axes[0].plot(full["global_epoch"], full[col], label=lbl, linestyle=ls)
    _add_stage_divider(axes[0]); axes[0].set_title("Validation Precision")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)
    for col, lbl, ls in [
        ("val_macro_recall", "Macro-R", "-"), ("val_micro_recall", "Micro-R", "--"),
        ("val_weighted_recall", "W-R", "-."), ("val_rare_recall", "Rare-R", ":"),
    ]:
        axes[1].plot(full["global_epoch"], full[col], label=lbl, linestyle=ls)
    _add_stage_divider(axes[1]); axes[1].set_title("Validation Recall")
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    p = os.path.join(OUT_DIR, "twostage_pr_curve.png")
    plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

    # ── 4. Per-stage rare-F1 comparison ────────────────────
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(s1["global_epoch"], s1["val_rare_f1"],
            color="royalblue", label="Stage 1 Rare-F1", marker="o", markersize=4)
    ax.plot(s2["global_epoch"], s2["val_rare_f1"],
            color="tomato",    label="Stage 2 Rare-F1", marker="s", markersize=4)
    if s1_end:
        ax.axvline(s1_end + 0.5, color="purple", linestyle=":", linewidth=1.5)
    ax.set_title("Rare-Class F1 Across Both Stages")
    ax.set_xlabel("Epoch (global)"); ax.set_ylabel("Rare-F1")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    p = os.path.join(OUT_DIR, "twostage_rare_f1.png")
    plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

    # ── 5. Epoch time ──────────────────────────────────────
    if "epoch_train_time_s" in full.columns:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar(s1["global_epoch"], s1["epoch_train_time_s"],
               color="royalblue", alpha=0.7, label="Stage 1")
        ax.bar(s2["global_epoch"], s2["epoch_train_time_s"],
               color="tomato",    alpha=0.7, label="Stage 2")
        ax.axhline(full["epoch_train_time_s"].mean(), color="black",
                   linestyle="--", label=f"Overall mean")
        ax.set_title("Per-Epoch Training Time")
        ax.set_xlabel("Epoch (global)"); ax.set_ylabel("Time (s)")
        ax.legend(); ax.grid(True, alpha=0.3, axis="y")
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "twostage_time_curve.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")


def save_confusion_matrix(cm, split_name, rare_labels=None):
    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABELS, yticklabels=LABELS,
                cmap="Blues", ax=ax)
    if rare_labels:
        for tick in ax.get_xticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
        for tick in ax.get_yticklabels():
            if tick.get_text() in rare_labels:
                tick.set_color("red")
    ax.set_title(f"{split_name.capitalize()} Confusion Matrix"
                 + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else ""))
    plt.tight_layout()
    p = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
    plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")


def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
    f1s    = [per_class_metrics[l]["f1"] for l in LABELS]
    colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
              for l in LABELS]
    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(LABELS, f1s, color=colors, edgecolor="white")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlim(0, 1.12); ax.set_xlabel("F1 Score")
    ax.set_title(f"{split_name.capitalize()} Per-Class F1"
                 + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)" if rare_labels else ""))
    ax.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    p = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
    plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")


# ═══════════════════════════════════════════════════════════
# RESULTS TABLE
# ═══════════════════════════════════════════════════════════
def print_metrics_table(dev_metrics, test_metrics,
                        s1_time=None, s2_time=None,
                        total_trainable=None, total_frozen=None):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare / Minority F1", "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Micro-Precision",    "micro_precision"),
        ("Weighted-Precision", "weighted_precision"),
        ("Rare-Precision",     "rare_precision"),
        ("Macro-Recall",       "macro_recall"),
        ("Micro-Recall",       "micro_recall"),
        ("Weighted-Recall",    "weighted_recall"),
        ("Rare-Recall",        "rare_recall"),
    ]
    print("\n" + "=" * 68)
    print("FINAL RESULTS  (InLegalBERT + BiLSTM + MHA + CRF  v5 Two-Stage)")
    print("=" * 68)
    if total_trainable:
        print(f"  Trainable Params : {total_trainable:,} | Frozen: {total_frozen:,}")
    if s1_time:
        print(f"  Stage-1 time     : {s1_time/60:.2f} min")
    if s2_time:
        print(f"  Stage-2 time     : {s2_time/60:.2f} min | "
              f"Total: {(s1_time+s2_time)/60:.2f} min")
    print("-" * 68)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 68)
    for label, key in rows:
        if key == "rare_f1":
            print("─" * 68)
        print(f"  {label:<28} {dev_metrics[key]:>12.4f} {test_metrics[key]:>12.4f}")
    print("=" * 68)
    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 62)
    print(f"  {'Label':<20} {'F1-Dev':>9} {'F1-Test':>9} {'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 62)
    for lbl in LABELS:
        dv = dev_metrics["per_class_metrics"][lbl]
        ts = test_metrics["per_class_metrics"][lbl]
        print(f"  {lbl:<20} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 62)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture : InLegalBERT → Sentence BiLSTM(128) → MHA(4) "
          "→ Context BiLSTM(64) → Linear → CRF")
    print("\n━━━ Two-Stage Training Strategy ━━━")
    print("  Stage 1 : CE-only | Strong balanced sampler | High LR | "
          f"≤{S1_EPOCHS} epochs")
    print("  Stage 2 : CRF+CE  | Mild balanced sampler  | Low LR  | "
          f"≤{S2_EPOCHS} epochs + SWA")

    # ── Load data ──────────────────────────────────────────
    train_raw  = load_jsonl(TRAIN_PATH)
    dev_raw    = load_jsonl(DEV_PATH)
    test_raw   = load_jsonl(TEST_PATH)
    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"\n  Train: {len(train_docs)} | Dev: {len(dev_docs)} | "
          f"Test: {len(test_docs)}")

    # ── Class analysis ─────────────────────────────────────
    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)
    pd.DataFrame([{"label": l, "frequency": label_freqs[l],
                   "is_rare": l in rare_labels} for l in LABELS]
    ).to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    # ── Class weights ──────────────────────────────────────
    cw = compute_class_weights(train_docs)
    # Stage 1: strong weights (amplified)
    cw_s1 = (cw ** 1.5).to(DEVICE)
    cw_s1 = cw_s1 / cw_s1.mean()
    # Stage 2: mild weights
    cw_s2 = cw.to(DEVICE)
    print("\n   📐 Stage-1 CE weights (^1.5):  ", end="")
    for i, lbl in enumerate(LABELS):
        print(f"{lbl}={cw_s1[i].item():.2f}", end=" ")
    print(f"\n   📐 Stage-2 CE weights (^1.0):  ", end="")
    for i, lbl in enumerate(LABELS):
        print(f"{lbl}={cw_s2[i].item():.2f}", end=" ")
    print()

    # ── Tokenizer & Datasets ───────────────────────────────
    print("\nLoading tokenizer...")
    tokenizer     = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)
    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    # ── Model ──────────────────────────────────────────────
    print("\nInitialising model...")
    model = InLegalBERT_BiLSTM_MHA_CRF(
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = S1_DROPOUT,       # Will be updated per stage
        freeze_layers    = BERT_FREEZE_LAYERS,
        class_weights_s1 = cw_s1,
        class_weights_s2 = cw_s2,
    )
    total_trainable, total_frozen, param_table = count_parameters(model)
    pd.DataFrame(param_table).to_csv(
        os.path.join(OUT_DIR, "parameter_summary.csv"), index=False)

    trainer = TwoStageTrainer(model, device=DEVICE)

    # ══════════════════════════════════════════════════════
    # STAGE 1
    # ══════════════════════════════════════════════════════
    s1_history, s1_time = trainer.train_stage1(
        train_dataset = train_dataset,
        dev_dataset   = dev_dataset,
        rare_ids      = rare_ids,
        docs          = train_docs,
        label_freqs   = label_freqs,
    )

    # Quick S1 evaluation snapshot
    print("\n--- Stage 1 final dev snapshot ---")
    s1_snap = trainer.evaluate(dev_dataset, rare_ids, split_name="s1_dev")
    print(f"  macro_f1={s1_snap['macro_f1']:.4f}  "
          f"weighted_f1={s1_snap['weighted_f1']:.4f}  "
          f"rare_f1={s1_snap['rare_f1']:.4f}")

    # ══════════════════════════════════════════════════════
    # STAGE 2
    # ══════════════════════════════════════════════════════
    s2_history, s2_time = trainer.train_stage2(
        train_dataset = train_dataset,
        dev_dataset   = dev_dataset,
        rare_ids      = rare_ids,
        docs          = train_docs,
        label_freqs   = label_freqs,
        tokenizer     = tokenizer,
    )

    # ── Combine & save history ─────────────────────────────
    hist_df = pd.DataFrame(s1_history + s2_history)
    hist_df.to_csv(os.path.join(OUT_DIR, "history.csv"), index=False)
    plot_history(hist_df)

    # ── Load best S2 checkpoint ────────────────────────────
    best_bin = os.path.join(BEST_MODEL_DIR, "pytorch_model.bin")
    if os.path.exists(best_bin):
        model.load_state_dict(torch.load(best_bin, map_location=DEVICE))
        model.set_stage(2)   # ensure CRF is active for evaluation
        print("\nLoaded best checkpoint.")

    # ── Dev evaluation ─────────────────────────────────────
    print("\nFinal evaluation on Dev set...")
    dev_metrics = trainer.evaluate(dev_dataset, rare_ids, split_name="dev",
                                   measure_inference_time=True)
    print(f"  Dev  Accuracy : {dev_metrics['accuracy']:.4f}")
    print(f"  Dev  Macro-F1 : {dev_metrics['macro_f1']:.4f}")
    print(f"  Dev  Rare-F1  : {dev_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA + CRF v5 Two-Stage\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(dev_metrics["cls_report"])

    save_confusion_matrix(dev_metrics["cm"], "dev", rare_labels)
    save_per_class_f1_chart(dev_metrics["per_class_metrics"], "dev", rare_labels)

    # ── Test evaluation ────────────────────────────────────
    print("\nFinal evaluation on Test set...")
    test_metrics = trainer.evaluate(test_dataset, rare_ids, split_name="test",
                                    measure_inference_time=True)
    print(f"  Test Accuracy : {test_metrics['accuracy']:.4f}")
    print(f"  Test Macro-F1 : {test_metrics['macro_f1']:.4f}")
    print(f"  Test Rare-F1  : {test_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA + CRF v5 Two-Stage\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_metrics["cls_report"])

    save_confusion_matrix(test_metrics["cm"], "test", rare_labels)
    save_per_class_f1_chart(test_metrics["per_class_metrics"], "test", rare_labels)

    pd.DataFrame({
        "true": [id2label[x] for x in test_metrics["all_trues"]],
        "pred": [id2label[x] for x in test_metrics["all_preds"]],
    }).to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    for split, mets in [("dev", dev_metrics), ("test", test_metrics)]:
        pd.DataFrame([{
            "label": lbl, "is_rare": lbl in rare_labels,
            "f1": mets["per_class_metrics"][lbl]["f1"],
            "precision": mets["per_class_metrics"][lbl]["precision"],
            "recall": mets["per_class_metrics"][lbl]["recall"],
        } for lbl in LABELS]).to_csv(
            os.path.join(OUT_DIR, f"{split}_per_class_metrics.csv"), index=False)

    # ── Summary JSON ───────────────────────────────────────
    scalar_keys = [
        "macro_f1","micro_f1","weighted_f1","rare_f1",
        "macro_precision","micro_precision","weighted_precision","rare_precision",
        "macro_recall","micro_recall","weighted_recall","rare_recall","accuracy",
    ]
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump({
            "model": {"name": "InLegalBERT+BiLSTM+MHA+CRF v5 Two-Stage",
                      "trainable_params": total_trainable, "frozen_params": total_frozen},
            "training": {
                "stage1": {"epochs": len(s1_history), "time_min": s1_time/60,
                           "es_metric": S1_ES_METRIC, "oversample_power": S1_OVERSAMPLE_POWER},
                "stage2": {"epochs": len(s2_history), "time_min": s2_time/60,
                           "es_metric": S2_ES_METRIC, "oversample_power": S2_OVERSAMPLE_POWER},
                "total_time_min": (s1_time + s2_time) / 60,
            },
            "rare_classes": rare_labels,
            "dev":  {k: dev_metrics[k]  for k in scalar_keys},
            "test": {k: test_metrics[k] for k in scalar_keys},
        }, f, indent=2)

    print_metrics_table(
        dev_metrics, test_metrics,
        s1_time=s1_time, s2_time=s2_time,
        total_trainable=total_trainable, total_frozen=total_frozen,
    )
    print(f"\n📁 All outputs → {OUT_DIR}/")


if __name__ == "__main__":
    main()

Device : cuda:0
Architecture : InLegalBERT → Sentence BiLSTM(128) → MHA(4) → Context BiLSTM(64) → Linear → CRF

━━━ Two-Stage Training Strategy ━━━
  Stage 1 : CE-only | Strong balanced sampler | High LR | ≤30 epochs
  Stage 2 : CRF+CE  | Mild balanced sampler  | Low LR  | ≤50 epochs + SWA

  Train: 245 | Dev: 30 | Test: 50

📊 Label frequency analysis (threshold ≤ 5%):
   PREAMBLE             14.50%  ( 4167 samples)
   FAC                  19.99%  ( 5744 samples)
   RLC                   2.62%  (  752 samples) ← RARE
   ISSUE                 1.28%  (  367 samples) ← RARE
   ARG_PETITIONER        4.58%  ( 1315 samples) ← RARE
   ARG_RESPONDENT        2.43%  (  698 samples) ← RARE
   ANALYSIS             36.66%  (10537 samples)
   STA                   1.67%  (  481 samples) ← RARE
   PRE_RELIED            4.97%  ( 1427 samples) ← RARE
   PRE_NOT_RELIED        0.55%  (  158 samples) ← RARE
   RATIO                 2.30%  (  661 samples) ← RARE
   RPC                   3.67%  ( 1055 sampl

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



❄️  BERT frozen: embeddings + layers 0-9.
🔥 BERT trainable: layers 10-11 (2 layers) + pooler.


MODEL PARAMETER SUMMARY  (v5 Two-Stage)
  Component                           Trainable     Frozen        Total
----------------------------------------------------------------------
  InLegalBERT Encoder                14,766,336 94,715,904  109,482,240
  Sentence BiLSTM                       919,552          0      919,552
  Multi-Head Attn Pooling               197,120          0      197,120
  Context BiLSTM                        164,864          0      164,864
  Classifier Head                         9,101          0        9,101
  CRF                                       195          0          195
──────────────────────────────────────────────────────────────────────
  ── TOTAL ──                        16,057,680 94,715,904  110,773,584

╔══════════════════════════════════════════════════════════════╗
║  STAGE 1 — Representation Learning (CE-only, Balanced)  ║
╚══════════════════